In [1]:
import pandas as pd
import time
import pickle
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import spotipy.exceptions

import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.config import SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET

In [12]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean.parquet")
CACHE_PATH = Path("spotify_cache.pkl")

SLEEP_TIME = 0.3
CHECKPOINT_EVERY = 100

In [13]:
# LOAD DATA
df = pd.read_parquet(DATA_PATH)

unique_tracks = df[["artist_clean", "track_clean"]].drop_duplicates().reset_index(drop=True)

unique_tracks["track_id"] = (
    unique_tracks["artist_clean"] + " - " + unique_tracks["track_clean"])

In [14]:
# LOAD CACHE
if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    print(f"Loaded cache with {len(cache)} entries")
else:
    cache = {}

In [15]:
# SPOTIFY CLIENT
sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET
    )
)

#results = sp.search(q="artist:deftones track:change", type="track", limit=1)
#print(results)
#track = results["tracks"]["items"][0]

# print({
#     "spotify_id": track["id"],
#     "artist": track["artists"][0]["name"],
#     "track": track["name"],
#     "popularity": track["popularity"]
# })

In [16]:
# SEARCH FUNCTION
# ========================

def search_spotify(sp, artist, track):
    query = f"artist:{artist} track:{track}"
    
    while True:
        try:
            results = sp.search(q=query, type="track", limit=1)
            items = results["tracks"]["items"]
            
            if not items:
                return None
            
            t = items[0]
            
            return {
                "spotify_id": t["id"],
                "spotify_artist": t["artists"][0]["name"],
                "spotify_track": t["name"],
                "popularity": t["popularity"],
            }
        
        except spotipy.exceptions.SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(e.headers.get("Retry-After", 5))
                print(f"Rate limited. Sleeping {retry_after}s")
                time.sleep(retry_after)
            else:
                return None


In [5]:
# MAIN LOOP

for i, row in unique_tracks.iterrows():
    key = row["track_id"]
    
    if key in cache:
        continue
    
    result = search_spotify(sp, row["artist_clean"], row["track_clean"])
    cache[key] = result
    
    if i % CHECKPOINT_EVERY == 0:
        print(f"{i} processed | cache size: {len(cache)}")
        
        with open(CACHE_PATH, "wb") as f:
            pickle.dump(cache, f)
    
    time.sleep(SLEEP_TIME)

In [17]:
# FINAL SAVE

with open(CACHE_PATH, "wb") as f:
    pickle.dump(cache, f)

print("Done")

Done


In [ ]:
spotify_df = pd.DataFrame.from_dict(cache, orient="index")
spotify_df.reset_index(inplace=True)
spotify_df.rename(columns={"index": "track_id"}, inplace=True)

In [ ]:
unique_tracks = unique_tracks.join(spotify_df)

In [ ]:
df = df.merge(unique_tracks, on=["artist_clean", "track_clean"], how="left")